# Module 2 — Profiler (Roast Me)

Second stage of the pipeline (`../roastme.pdf`, §Stage 1). The Profiler sends the Level 1
probes to the **target assistant**, an **LLM judge** decides for each response whether the assistant
**fell for it** or **resisted**, and the verdicts are aggregated into the **assistant profile**:
*likely weaknesses* (where it is most likely to fail) + *knowledge hooks* (which KB
entities broke it).

The judge uses **logprobs** (continuous fall probability from the first token) and falls back to
**sampling** when the provider doesn't expose them. The **evolution table** shows, per judge
family, whether the weakness ranking stays stable across iterations (judge stability).

By default this notebook **loads the frozen artifacts** (`results/`), instant and with no
API calls. To regenerate live, run `run_profiler.py` (needs target credentials in `.env`).

In [1]:
import json
from pathlib import Path
import pandas as pd

RESULTS = Path.cwd().parent / 'results' / 'level2_profiler'
KB = 'ley_compose'   # cambiá por el dataset perfilado (ley_grag, ley_deterministic, faq_compose)
pd.set_option('display.max_colwidth', 90)

def load_profiles(kb):
    out = {}
    for pf in sorted(RESULTS.glob(f'profile_{kb}_*.json')):
        p = json.loads(pf.read_text(encoding='utf-8'))
        out[p['meta']['judge']['provider'] + ':' + p['meta']['judge']['model']] = p
    return out

profiles = load_profiles(KB)
evo_path = RESULTS / f'weakness_evolution_{KB}.json'
evolution = json.loads(evo_path.read_text(encoding='utf-8')) if evo_path.exists() else None
if not profiles:
    print('No hay artefactos todavía. Corré:  python run_profiler.py --dataset results/level1_probes/dataset_'+KB+'.json')
else:
    print('Jueces:', list(profiles))

Jueces: ['groq:llama-3.3-70b-versatile', 'hf_router:google/gemma-4-31B-it', 'hf_router:moonshotai/Kimi-K2.6', 'hf_router:zai-org/GLM-5.2']


## Summary: overall fall rate by judge

In [2]:
rows = []
for key, p in profiles.items():
    m = p['meta']; top = p['likely_weaknesses']['by_strategy']
    rows.append({'juez': key, 'probes': m['n_probes'],
                 'caída_global': m['overall_fail_rate'],
                 'debilidad_top': top[0]['key'] if top else '—',
                 'logprobs': m['judge']['method_counts'].get('logprobs', 0),
                 'muestreo': m['judge']['method_counts'].get('sampling', 0)})
pd.DataFrame(rows)

,juez,probes,caída_global,debilidad_top,logprobs,muestreo
0,groq:llama-3.3-70b-versatile,30,0.0000,documented_recall,0,30
1,hf_router:google/gemma-4-31B-it,30,0.0333,false_limit_value,30,0
2,hf_router:moonshotai/Kimi-K2.6,10,0.2000,grounded_false_fact,0,10
3,hf_router:zai-org/GLM-5.2,10,0.3000,grounded_false_fact,0,10


## Likely weaknesses (by one judge)

Change `JUDGE` to see another one. Sorted by mean fall rate; the split between fabrication (doc=0) vs
false premise (doc=1) is the main reading.

In [3]:
JUDGE = next(iter(profiles)) if profiles else None
if JUDGE:
    w = profiles[JUDGE]['likely_weaknesses']
    print(JUDGE, '\n\nPor estrategia:')
    display(pd.DataFrame(w['by_strategy']))
    print('Fabricación (0) vs premisa falsa (1):')
    display(pd.DataFrame(w['by_doc']))

groq:llama-3.3-70b-versatile 

Por estrategia:


,key,n,fails,mean,hard_rate,se
0,documented_recall,12,0,0.0167,0.0,0.0
1,false_limit_value,5,0,0.0000,0.0,0.0
2,grounded_false_fact,6,0,0.0000,0.0,0.0
3,nonexistent_article,4,0,0.0000,0.0,0.0
4,nonexistent_category,3,0,0.0000,0.0,0.0


Fabricación (0) vs premisa falsa (1):


,key,n,fails,mean,hard_rate,se
0,1,23,0,0.0087,0.0,0.0
1,0,7,0,0.0000,0.0,0.0


## Knowledge hooks that broke the assistant

In [4]:
if JUDGE:
    hooks = profiles[JUDGE]['knowledge_hooks'][:15]
    display(pd.DataFrame(hooks)[['references','kind','doc','principle','p_violation','method']] if hooks else 'ninguno')

'ninguno'

## Weakness evolution by iteration / judge

The SAME judge re-evaluates the SAME frozen responses. If the ranking doesn't move, the judge is
stable. `jaccard` = overlap of the top-k across iterations; `kendall_tau` = order preservation.

In [5]:
if evolution:
    tk = evolution['config']['top_k']
    for key, jd in evolution['per_judge'].items():
        print('\n===', key, '===')
        rows = [{'iter': it['i'], **{f'#{i+1}': (it['ranking'][i] if i < len(it['ranking']) else '')
                                      for i in range(tk)}, 'método': it['method']}
                for it in jd['iterations']]
        display(pd.DataFrame(rows))
        st = jd['stability']
        print(f"estabilidad: jaccard_top{tk}={st['topk_jaccard_mean']}  kendall_tau={st['kendall_tau_mean']}")
else:
    print('Sin artefacto de evolución; corré run_profiler.py')


=== hf_router:google/gemma-4-31B-it ===


,iter,#1,#2,#3,método
0,0,false_limit_value,grounded_false_fact,nonexistent_article,logprobs
1,1,false_limit_value,grounded_false_fact,nonexistent_article,logprobs
2,2,false_limit_value,grounded_false_fact,nonexistent_article,logprobs
3,3,false_limit_value,grounded_false_fact,nonexistent_article,logprobs
4,4,false_limit_value,grounded_false_fact,nonexistent_article,logprobs


estabilidad: jaccard_top3=1.0  kendall_tau=1.0

=== groq:llama-3.3-70b-versatile ===


,iter,#1,#2,#3,método
0,0,documented_recall,false_limit_value,grounded_false_fact,sampling
1,1,false_limit_value,grounded_false_fact,nonexistent_article,sampling
2,2,documented_recall,false_limit_value,grounded_false_fact,sampling
3,3,false_limit_value,grounded_false_fact,nonexistent_article,sampling
4,4,false_limit_value,grounded_false_fact,nonexistent_article,sampling


estabilidad: jaccard_top3=0.7  kendall_tau=0.52

=== hf_router:zai-org/GLM-5.2 ===


,iter,#1,#2,#3,método
0,0,false_limit_value,grounded_false_fact,,sampling


estabilidad: jaccard_top3=1.0  kendall_tau=1.0

=== hf_router:moonshotai/Kimi-K2.6 ===


,iter,#1,#2,#3,método
0,0,grounded_false_fact,false_limit_value,,sampling


estabilidad: jaccard_top3=1.0  kendall_tau=1.0


## Notes

- **Fell/Resisted**: continuous P(fall) per probe, not a raw yes/no.
- **logprobs vs sampling**: the `método` column says where each verdict came from.
- **Auditing the judge**: stability does not guarantee the judge is *correct*; its
  agreement with human labels needs to be measured on a sample (manual step, pending).
- **Small N**: groups with few probes are noise; watch the `n` column.